# Population structurée par âge

| Module | Durée | Prérequis |
|---|---|---|
| M7 – Valeurs propres et dynamique | 75 min | M5 |

In [ ]:
# Style graphique Liouaï : couleurs du logo, axes sobres
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

LIOUAI = {"navy": "#243743", "vert": "#28B78D", "vert_texte": "#167A5C", "gris": "#A1A9AE",
          "gris_texte": "#5C6870", "jaune": "#E5C800", "fond": "#F3F5F6"}
plt.rcParams.update({
    "axes.prop_cycle": plt.cycler(color=[LIOUAI["vert"], LIOUAI["navy"], LIOUAI["gris"],
                                         LIOUAI["jaune"], LIOUAI["vert_texte"], LIOUAI["gris_texte"]]),
    "axes.edgecolor": LIOUAI["gris"], "axes.labelcolor": LIOUAI["navy"],
    "axes.titlecolor": LIOUAI["navy"], "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": LIOUAI["gris_texte"], "ytick.color": LIOUAI["gris_texte"],
    "text.color": LIOUAI["navy"], "grid.color": "#E1E6E9", "legend.frameon": False,
    "font.family": "sans-serif", "font.sans-serif": ["Roboto", "Helvetica", "Arial", "DejaVu Sans"],
    "figure.dpi": 100,
})
# Dégradé de marque : gris clair -> vert -> navy (et sa version inversée)
if "liouai" not in mpl.colormaps:
    _cmap = LinearSegmentedColormap.from_list("liouai", [LIOUAI["fond"], LIOUAI["vert"], LIOUAI["navy"]])
    mpl.colormaps.register(_cmap)
    mpl.colormaps.register(_cmap.reversed())

## ① Définir la question

Une association de protection de la nature suit depuis dix ans une petite population de **chouettes chevêches** dans un bocage. Elle compte chaque printemps les femelles (les mâles suivent), réparties en trois classes d'âge : les juvéniles de l'année, les oiseaux d'un an, et les adultes de deux ans et plus.

Le dernier recensement donne 40 juvéniles, 12 oiseaux d'un an et 30 adultes. Les observations des années passées ont permis d'estimer :

- la **survie** d'une année à l'autre : 20 % pour les juvéniles, 50 % pour les oiseaux d'un an, 60 % pour les adultes ;
- la **fécondité** : en moyenne 0,9 jeune femelle par femelle d'un an, et 1,6 par adulte.

Ces valeurs sont fictives, mais du même ordre que celles des études de terrain. La survie des juvéniles est faible, car beaucoup de jeunes meurent faute de cavités pour s'abriter. L'association envisage de poser des **nichoirs** pour l'améliorer.

**Les questions : la population va-t-elle croître ou décliner à long terme, et à quel rythme ? Quelle survie des juvéniles faudrait-il atteindre pour la stabiliser ?**

**Activité** : Quelles hypothèses ce modèle fait-il sur la vie des chouettes ? Lesquelles vous semblent les plus discutables ?

Indiquez vos réponses et justifiez-les.

## ② Estimer avant de calculer

In [ ]:
pari_effectif_dans_20_ans = 80      # nombre total de femelles dans 20 ans (modifiez)
pari_survie_juvenile_necessaire = 0.3   # pour que la population se maintienne (modifiez)

## ③ Abstraire : la matrice de Leslie

On range les effectifs dans un vecteur $\mathbf{n}_t = (n_0\,;\,n_1\,;\,n_2)$. D'une année à l'autre, les naissances alimentent la première classe, et les survivants passent dans la classe suivante :

![Schéma : cycle de vie à trois classes ; les survies s0, s1, s2 font passer d'une classe à la suivante, les fécondités f1 et f2 ramènent vers les juvéniles ; la matrice de Leslie regroupe ces coefficients](../../assets/schemas/schema-leslie.svg)

$$
\mathbf{n}_{t+1} = L\,\mathbf{n}_t
\qquad\text{donc}\qquad
\mathbf{n}_t = L^t\,\mathbf{n}_0 .
$$

**Pourquoi les valeurs propres ?** Si la population de départ était exactement un vecteur propre $\mathbf{v}$ de $L$, chaque année la multiplierait simplement par $\lambda$ : $\mathbf{n}_t = \lambda^t \mathbf{v}$. En général, $\mathbf{n}_0$ est une combinaison de vecteurs propres, et c'est la valeur propre de plus grand module qui l'emporte (notebook 7.1). À long terme :

$$
\mathbf{n}_t \approx c\,\lambda_1^t\,\mathbf{v}_1 .
$$

- $\lambda_1$ est le **taux de croissance** annuel : la population croît si $\lambda_1 > 1$, décline si $\lambda_1 < 1$ ;
- $\mathbf{v}_1$, ramené à une somme de 1, donne la **structure d'âge stable** : la proportion de chaque classe.

**Activité** : Pour une espèce à deux classes, de matrice $L = \begin{pmatrix} 0{,}5 & 2 \\ 0{,}3 & 0 \end{pmatrix}$, calculez le polynôme caractéristique et la valeur propre positive. La population croît-elle ? De combien de pour cent par an ?

Indiquez vos réponses et justifiez-les.

## ④ Calculer

### Simuler trente ans

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def leslie(s0=0.2, s1=0.5, s2=0.6, f1=0.9, f2=1.6):
    return np.array([[0, f1, f2],
                     [s0, 0, 0],
                     [0, s1, s2]])

L = leslie()
n0 = np.array([40., 12, 30])
annees = 31
effectifs = np.zeros((annees, 3))
effectifs[0] = n0
for t in range(1, annees):
    effectifs[t] = L @ effectifs[t - 1]

total = effectifs.sum(axis=1)
print(f"Effectif total : {total[0]:.0f} aujourd'hui, {total[10]:.1f} dans 10 ans, {total[20]:.1f} dans 20 ans")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for k, nom in enumerate(["juvéniles", "1 an", "adultes"]):
    axes[0].semilogy(effectifs[:, k], label=nom)
axes[0].semilogy(total, "k--", lw=1, label="total")
axes[0].set_xlabel("années"); axes[0].set_ylabel("femelles (échelle log)"); axes[0].legend(); axes[0].set_title("Effectifs")
taux = total[1:] / total[:-1]
axes[1].plot(range(1, annees), taux, "o-", ms=3, color=LIOUAI["navy"])
axes[1].set_xlabel("années"); axes[1].set_ylabel("effectif(t) / effectif(t-1)"); axes[1].set_title("Taux de croissance annuel")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Proportions par classe après 30 ans :", (effectifs[-1] / total[-1]).round(3))

**Activité** : Sur un graphique en échelle logarithmique, une croissance (ou une décroissance) géométrique apparaît comme une droite. Les courbes le deviennent-elles ? Au bout de combien d'années ? Le taux de croissance se stabilise-t-il ?

Indiquez vos réponses et justifiez-les.

### Les éléments propres de $L$

In [ ]:
def element_dominant(L):
    valeurs, vecteurs = np.linalg.eig(L)
    k = np.argmax(np.abs(valeurs))
    lam = valeurs[k].real
    v = vecteurs[:, k].real
    return lam, v / v.sum()            # structure d'âge : proportions de somme 1

lam, structure = element_dominant(L)
print("Toutes les valeurs propres :", np.round(np.linalg.eigvals(L), 3))
print(f"Valeur propre dominante : {lam:.4f}, soit {100 * (lam - 1):+.1f} % par an")
print("Structure d'âge stable   :", structure.round(3))

### Quelle survie des juvéniles faut-il atteindre ?

On transforme la constante $s_0$ en **paramètre**, et l'on trace $\lambda_1$ en fonction de $s_0$.

In [ ]:
s0_valeurs = np.linspace(0.05, 0.6, 111)
lambdas = np.array([element_dominant(leslie(s0=s))[0] for s in s0_valeurs])
s0_critique = np.interp(1.0, lambdas, s0_valeurs)         # lambda croît avec s0 : on peut interpoler

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(s0_valeurs, lambdas, color=LIOUAI["navy"], lw=2)
ax.axhline(1, color=LIOUAI["gris"], ls="--")
ax.axvline(0.2, color=LIOUAI["jaune"], label="survie actuelle")
ax.axvline(s0_critique, color=LIOUAI["vert"], label=f"seuil de maintien : {s0_critique:.3f}")
ax.set_xlabel("survie des juvéniles s0"); ax.set_ylabel("taux de croissance λ1"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Où agir ? Comparer les leviers

Une amélioration de 10 % de chaque paramètre, pris un par un, ne change pas $\lambda_1$ de la même façon.

In [ ]:
reference = dict(s0=0.2, s1=0.5, s2=0.6, f1=0.9, f2=1.6)
lam_ref = element_dominant(leslie(**reference))[0]
print(f"{'paramètre':10s} {'λ1 après +10 %':>15s} {'gain':>8s}")
for nom, valeur in reference.items():
    lam_nouveau = element_dominant(leslie(**{**reference, nom: 1.1 * valeur}))[0]
    print(f"{nom:10s} {lam_nouveau:15.4f} {lam_nouveau - lam_ref:+8.4f}")

## ⑤ Interpréter

In [ ]:
print(f"Vos paris : {pari_effectif_dans_20_ans} femelles dans 20 ans ; survie juvénile nécessaire {pari_survie_juvenile_necessaire}")
print(f"Résultats : {total[20]:.0f} femelles dans 20 ans ; survie juvénile nécessaire {s0_critique:.2f}")

**Activité** :

1. Rédigez pour l'association un message de quelques phrases : avenir de la population sans action, objectif chiffré pour les nichoirs.
2. Que signifie concrètement la structure d'âge stable ? Comment l'association pourrait-elle la vérifier sur le terrain ?
3. Le tableau des leviers montre que tous les paramètres ne se valent pas. Lequel est le plus efficace ? Est-il pour autant le plus facile à améliorer ?
4. Pourquoi les deux autres valeurs propres, complexes, n'ont-elles aucune influence à long terme ?

Indiquez vos réponses et justifiez-les.

## ⑥ Critiquer et généraliser

- **Des paramètres constants ?** Les survies varient d'une année à l'autre (hivers rigoureux, disponibilité des proies). Un taux moyen légèrement inférieur à 1 conduit-il à une extinction certaine ? En combien de temps ?
- **Pas de limite ?** Avec $\lambda_1 > 1$, le modèle prévoit une croissance sans fin. Qu'est-ce qui l'arrêterait en réalité ? Le modèle reste-t-il linéaire ?
- **Vérifier autrement.** Pour cette matrice, la condition $\lambda_1 = 1$ s'écrit exactement $s_0 = \dfrac{1 - s_2}{f_1(1 - s_2) + f_2 s_1}$. Vérifiez-la à partir de $\det(L - I) = 0$, puis comparez au seuil trouvé par le calcul.
- **Généraliser.** Écrivez une fonction `risque_extinction(s0_moyen, annees=50, essais=2000)` qui simule la population en tirant chaque année la survie des juvéniles au hasard entre $s_0 - 0{,}1$ et $s_0 + 0{,}1$, et renvoie la proportion d'essais où l'effectif total tombe sous 10 femelles. Comparez la situation actuelle et la situation avec nichoirs ($s_0 = 0{,}35$).

In [ ]:
def risque_extinction(s0_moyen, annees=50, essais=2000):
    # À compléter
    ...

## Corrigé

*À n'ouvrir qu'après avoir cherché.*

In [ ]:
r = reference
formule = (1 - r["s2"]) / (r["f1"] * (1 - r["s2"]) + r["f2"] * r["s1"])
print(f"Seuil exact par la formule : {formule:.4f} ; par interpolation : {s0_critique:.4f}")

def risque_extinction(s0_moyen, annees=50, essais=2000, graine=1):
    rng = np.random.default_rng(graine)
    disparitions = 0
    for _ in range(essais):
        n = n0.copy()
        for _ in range(annees):
            s0 = rng.uniform(s0_moyen - 0.1, s0_moyen + 0.1)
            n = leslie(s0=s0) @ n
            if n.sum() < 10:
                disparitions += 1
                break
    return disparitions / essais

for s0 in (0.2, 0.3, 0.35, 0.45):
    print(f"s0 moyen = {s0:.2f} : λ1 = {element_dominant(leslie(s0=s0))[0]:.3f}, "
          f"risque de tomber sous 10 femelles en 50 ans = {risque_extinction(s0):.0%}")

## Bilan — auto-évaluation

Pour chaque compétence travaillée dans ce notebook, choisissez un niveau : **0** non observé · **1** guidé · **2** autonome · **3** transfert.
La cellule enregistre votre bilan dans `progression.json`, à la racine du site.

In [ ]:
import json, datetime, pathlib

bilan = {
    "Relier les traits du résultat à leur sens réel": 1,
    "Repérer les traits intéressants du résultat": 1,
    "Identifier, énoncer et justifier les hypothèses": 1,
    "Énoncer le résultat dans le contexte du problème": 1,
    "Transformer des constantes en paramètres": 1,
}   # modifiez les niveaux

ici = pathlib.Path.cwd()
racine = next((p for p in [ici, *ici.parents] if (p / "myst.yml").exists()), ici)
fichier = racine / "progression.json"
historique = json.loads(fichier.read_text(encoding="utf-8")) if fichier.exists() else []
historique.append({"notebook": "7.2-population-leslie", "date": datetime.date.today().isoformat(), "niveaux": bilan})
fichier.write_text(json.dumps(historique, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Bilan enregistré ({len(historique)} entrée(s) dans {fichier.name}).")